<a href="https://colab.research.google.com/github/1stzl01/Reserving/blob/main/ICR_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Micro-Level Stochastic Claims Reserving (ICR Framework)

> **Methodology & Attribution:**
> The micro-level simulation engine implemented in this notebook is based on the **Individual Claim Reserving (ICR)** method.
>
> *Reference:* Chalnot, J.B., Gremillet, M., & Miehe, P. (2015). *Implementing the Individual Claims Reserving Method, a new approach in non-life reserving*. ASTIN Bulletin. Further exposition follows Descamps, H. (2017), *Étude de méthodes de provisionnement individuelles et application sur un portefeuille Auto*, ISFA.

This notebook implements the **Individual Claim Reserving (ICR)** methodology for non-life insurance. Unlike classical deterministic methods (e.g., Chain Ladder) that aggregate data into run-off triangles, this micro-level approach uses the timeline of every individual claim, reducing aggregation bias at the cost of stronger distributional assumptions.

**Methodology:**
1. **Data Preparation**: Process individual claim logs capturing occurrence, notification, payments, and closure.
2. **IBNR Frequency**: Estimate the number of Incurred But Not Reported (IBNR) claims via a Poisson process whose rate is calibrated empirically as historical late-reported claims divided by exposure.
3. **Event Probabilities**: Calculate empirical probabilities `P(E_j = m)` for the 4-state event model on open claims — Closure without payment (Type 1), Closure with payment (Type 2), Intermediate payment (Type 3), and an "Empty" event (Type 4).
4. **Severity Calibration**: Fit a Gamma distribution per development year to historical paid events (Types 2 and 3) to model future claim severities.
5. **Stochastic Simulation Engine**: Project every open claim (RBNS) and simulated IBNR forward period-by-period until closure, collecting the resulting reserve realisations across Monte Carlo paths.

*Note: This is an illustrative implementation. Some extensions described in the source papers — multinomial-logit event probabilities, tail factors, initial-reserve-category segmentation — are not included.*


In [1]:
# =====================================================================
# SECTION 1: HISTORICAL DATA SIMULATION
# =====================================================================
# Event types follow the ICR framework:
#   Type 1: Closure without payment       (terminal, no payment)
#   Type 2: Closure with payment          (terminal, payment)
#   Type 3: Intermediate payment          (continues, payment)
#   Type 4: "Empty" event                 (continues, no payment)
#
# Key fix: claim lifetime is no longer drawn independently of event
# types. A claim's payment history terminates the first time a Type 1
# or Type 2 event is drawn. This makes the synthetic historical data
# internally consistent with the closure logic enforced in Section 5.

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import gamma, poisson

np.random.seed(42)
num_claims = 10_000

ORIGIN_YEAR_MIN = 2006
ORIGIN_YEAR_MAX = 2016
VALUATION_YEAR = ORIGIN_YEAR_MAX
MAX_DEV_YEARS = 11

# Per-dev-year event probabilities. Expected claim lifetime is
# 1 / (p1 + p2) = 2.5 dev years, leaving a meaningful tail.
EVENT_TYPES = (1, 2, 3, 4)
EVENT_PROBS_GEN = [0.20, 0.20, 0.20, 0.40]
SEVERITY_SHAPE, SEVERITY_SCALE = 2, 1000

data = {
    'claim_id': [f"CLM_{i}" for i in range(num_claims)],
    'occurrence_year': np.random.randint(ORIGIN_YEAR_MIN, ORIGIN_YEAR_MAX + 1, size=num_claims),
    'reporting_delay': np.random.choice((1, 2, 3, 4), p=[0.7, 0.2, 0.08, 0.02], size=num_claims),
}
df_claims = pd.DataFrame(data)
df_claims['reporting_year'] = df_claims['occurrence_year'] + df_claims['reporting_delay'] - 1

payments = []
statuses = []

for _, row in df_claims.iterrows():
    claim_id = row['claim_id']
    reporting_year = row['reporting_year']
    closed = False

    for dev in range(1, MAX_DEV_YEARS + 1):
        event_type = np.random.choice(EVENT_TYPES, p=EVENT_PROBS_GEN)
        payment_amt = (gamma.rvs(a=SEVERITY_SHAPE, scale=SEVERITY_SCALE)
                       if event_type in (2, 3) else 0.0)

        calendar_year = reporting_year + dev - 1
        if calendar_year > VALUATION_YEAR:
            break

        payments.append({
            'claim_id': claim_id,
            'dev_year': dev,
            'calendar_year': calendar_year,
            'event_type': event_type,
            'payment_amt': payment_amt,
        })

        if event_type in (1, 2):
            closed = True
            break

    statuses.append('Closed' if closed else 'Open')

df_claims['status'] = statuses
df_payments = pd.DataFrame(payments)

print(f"Simulated {len(df_claims)} claims; {len(df_payments)} payment events.")
print(f"Open (RBNS): {(df_claims['status'] == 'Open').sum()}, "
      f"Closed: {(df_claims['status'] == 'Closed').sum()}")

Simulated 10000 claims; 20794 payment events.
Open (RBNS): 1751, Closed: 8249


In [3]:
# =====================================================================
# SECTION 2: DIAGNOSTIC POISSON GLM FOR REPORTING FREQUENCY
# =====================================================================
# The canonical ICR estimator (Descamps 2.3.1) is empirical and lives
# in Section 4. This GLM is purely diagnostic: it treats reporting_delay
# as CATEGORICAL (each delay gets its own parameter), includes
# occurrence_year as a factor, and is fit ONLY on fully-observable cells
# (i + j - 1 <= valuation_year) to avoid right-truncation bias.

exposure_data = {
    2006: 165212, 2007: 160259, 2008: 158331, 2009: 162072, 2010: 169866,
    2011: 151725, 2012: 139566, 2013: 139168, 2014: 138332, 2015: 135138,
    2016: 134707,
}

ibnr_agg = (df_claims
            .groupby(['occurrence_year', 'reporting_delay'])
            .size()
            .reset_index(name='claim_count'))
ibnr_agg['exposure'] = ibnr_agg['occurrence_year'].map(exposure_data)
ibnr_agg['fully_observed'] = (
    ibnr_agg['occurrence_year'] + ibnr_agg['reporting_delay'] - 1 <= VALUATION_YEAR
)
ibnr_fit = ibnr_agg[ibnr_agg['fully_observed']].copy()

poisson_model = smf.glm(
    formula='claim_count ~ C(reporting_delay) + C(occurrence_year)',
    data=ibnr_fit,
    family=sm.families.Poisson(),
    offset=np.log(ibnr_fit['exposure']),
).fit()

print("\n--- Diagnostic Poisson GLM (Section 2) ---")
print(poisson_model.summary())


--- Diagnostic Poisson GLM (Section 2) ---
                 Generalized Linear Model Regression Results                  
Dep. Variable:            claim_count   No. Observations:                   38
Model:                            GLM   Df Residuals:                       24
Model Family:                 Poisson   Df Model:                           13
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -136.41
Date:                Sat, 25 Apr 2026   Deviance:                       18.274
Time:                        13:39:46   Pearson chi2:                     18.7
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

In [4]:
# =====================================================================
# SECTION 3: EVENT PROBABILITIES BY DEVELOPMENT YEAR
# =====================================================================
# Empirical P(E_j = m) for j = development year, m = event type.
# Laplace smoothing guards against zero probabilities at sparse late
# development years.

LAPLACE_ALPHA = 1.0

event_counts = (df_payments
                .groupby(['dev_year', 'event_type'])
                .size()
                .unstack(fill_value=0)
                .reindex(columns=EVENT_TYPES, fill_value=0))

K = len(EVENT_TYPES)
event_probs = (event_counts + LAPLACE_ALPHA).div(
    event_counts.sum(axis=1) + K * LAPLACE_ALPHA, axis=0
)
event_probs_dict = event_probs.to_dict(orient='index')

print(f"\n--- Section 3: P(E_j = m) ---")
print(f"Observations per dev_year: {event_counts.sum(axis=1).to_dict()}")
print(event_probs.round(4))


--- Section 3: P(E_j = m) ---
Observations per dev_year: {1: 9603, 2: 5202, 3: 2826, 4: 1522, 5: 806, 6: 439, 7: 218, 8: 106, 9: 51, 10: 18, 11: 3}
event_type       1       2       3       4
dev_year                                  
1           0.2041  0.2015  0.1987  0.3956
2           0.1980  0.1923  0.1994  0.4103
3           0.1933  0.2025  0.2102  0.3940
4           0.1920  0.2025  0.2031  0.4024
5           0.1704  0.1864  0.2259  0.4173
6           0.2009  0.1851  0.2235  0.3905
7           0.2117  0.1757  0.1667  0.4459
8           0.1818  0.1818  0.2000  0.4364
9           0.1273  0.2182  0.2000  0.4545
10          0.3182  0.2273  0.1364  0.3182
11          0.1429  0.2857  0.1429  0.4286


In [5]:
# =====================================================================
# SECTION 4: IBNR FREQUENCY CALIBRATION + SIMULATION
# =====================================================================
# Frequency formula: lambda_tilde_j = (sum N_tj) / (sum E_t)
# where the sums run over origin years for which delay j is OBSERVABLE
# (i + j - 1 <= valuation_year). Using total exposure across all years
# as the denominator (as in the original draft) systematically
# under-estimates lambda for long delays.

max_delay = int(df_claims['reporting_delay'].max())

counts = (df_claims
          .groupby(['occurrence_year', 'reporting_delay'])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=range(1, max_delay + 1), fill_value=0))

ibnr_frequencies = {}
for j in range(1, max_delay + 1):
    observable_years = [i for i in counts.index if i + j - 1 <= VALUATION_YEAR]
    if not observable_years:
        ibnr_frequencies[j] = 0.0
        continue
    numerator = counts.loc[observable_years, j].sum()
    denominator = sum(exposure_data[i] for i in observable_years)
    ibnr_frequencies[j] = numerator / denominator if denominator > 0 else 0.0

categories = ('Default',)
category_probs = (1.0,)

simulated_ibnrs = []
ibnr_counter = 0

for p in range(VALUATION_YEAR + 1, VALUATION_YEAR + max_delay + 1):
    for i in counts.index:
        delay = p - i + 1
        if delay < 1 or delay > max_delay:
            continue
        if i + delay - 1 <= VALUATION_YEAR:
            continue
        e_i = exposure_data.get(i)
        if e_i is None:
            continue
        expected = e_i * ibnr_frequencies[delay]
        n = int(poisson.rvs(mu=expected))
        for _ in range(n):
            ibnr_counter += 1
            simulated_ibnrs.append({
                'claim_id': f"IBNR_{ibnr_counter:07d}",
                'occurrence_year': i,
                'reporting_delay': delay,
                'reporting_year': p,
                'status': 'Open',
                'initial_reserve_category': np.random.choice(categories, p=category_probs),
            })

df_simulated_ibnrs = pd.DataFrame(simulated_ibnrs)

rbns = df_claims[df_claims['status'] == 'Open'].copy()
rbns['initial_reserve_category'] = 'Default'

ibnr_cols = ['claim_id', 'occurrence_year', 'reporting_delay',
             'reporting_year', 'status', 'initial_reserve_category']

if len(df_simulated_ibnrs) > 0:
    open_claims = pd.concat([rbns[ibnr_cols], df_simulated_ibnrs[ibnr_cols]],
                            ignore_index=True)
else:
    open_claims = rbns[ibnr_cols].copy()

print(f"\n--- Section 4: IBNR ---")
print(f"lambda_tilde_j: {{j: round(v, 6) for j, v in ibnr_frequencies.items()}}")
print(f"Simulated {len(df_simulated_ibnrs)} IBNR claims; "
      f"open frame = {len(rbns)} RBNS + {len(df_simulated_ibnrs)} IBNR "
      f"= {len(open_claims)}.")


--- Section 4: IBNR ---
lambda_tilde_j: {j: round(v, 6) for j, v in ibnr_frequencies.items()}
Simulated 334 IBNR claims; open frame = 1751 RBNS + 334 IBNR = 2085.


In [6]:
# =====================================================================
# SECTION 4b: GAMMA SEVERITY CALIBRATION
# =====================================================================
# Per-dev-year Gamma fit on non-zero payments (event types 2 and 3).
# Skipped dev years fall back to a global pooled estimate so the
# simulation never raises KeyError.

MIN_OBS_FOR_FIT = 10

paid_events = df_payments[df_payments['event_type'].isin((2, 3))]

severity_params = {}
skipped_dev_years = []

for dev_year, group in paid_events.groupby('dev_year'):
    arr = group['payment_amt'].values
    if len(arr) > MIN_OBS_FOR_FIT:
        shape, _, scale = gamma.fit(arr, floc=0)
        severity_params[int(dev_year)] = {'shape': shape, 'scale': scale}
    else:
        skipped_dev_years.append((int(dev_year), len(arr)))

pooled = paid_events['payment_amt'].values
if len(pooled) > 0:
    p_shape, _, p_scale = gamma.fit(pooled, floc=0)
    severity_params_pooled = {'shape': p_shape, 'scale': p_scale}
else:
    severity_params_pooled = {'shape': 2.0, 'scale': 1000.0}

print(f"\n--- Section 4b: Severity ---")
for d in sorted(severity_params):
    s = severity_params[d]
    print(f"  dev {d}: shape={s['shape']:.3f}, scale={s['scale']:.1f}")
if skipped_dev_years:
    print(f"  skipped (use pooled): {skipped_dev_years}")
print(f"  pooled fallback: shape={severity_params_pooled['shape']:.3f}, "
      f"scale={severity_params_pooled['scale']:.1f}")


--- Section 4b: Severity ---
  dev 1: shape=2.015, scale=994.8
  dev 2: shape=2.006, scale=1016.8
  dev 3: shape=2.002, scale=997.9
  dev 4: shape=1.948, scale=1008.7
  dev 5: shape=1.985, scale=1014.7
  dev 6: shape=2.080, scale=1037.0
  dev 7: shape=1.908, scale=1163.3
  dev 8: shape=1.693, scale=1109.8
  dev 9: shape=1.817, scale=841.3
  skipped (use pooled): [(10, 6), (11, 1)]
  pooled fallback: shape=2.001, scale=1005.9


In [7]:
# =====================================================================
# SECTION 5: MONTE CARLO STOCHASTIC-RESERVING ENGINE
# =====================================================================
# For each open claim, project forward period-by-period using the
# calibrated event probabilities and severity. The state machine
# enforced here is:
#   Type 1: closure no payment      -> claim closed
#   Type 2: closure with payment    -> claim closed, sample severity
#   Type 3: intermediate payment    -> continue, sample severity
#   Type 4: empty event             -> continue, no payment
#
# Wrapped in an outer Monte Carlo loop so the OUTPUT is a distribution
# (mean, std, percentiles, 99.5% VaR), not a single point estimate.

N_SIMULATIONS = 500
MAX_DEV = MAX_DEV_YEARS
EVENT_TYPES_ARR = np.array([1, 2, 3, 4])
RNG = np.random.default_rng(2024)

last_dev_observed = (df_payments
                     .groupby('claim_id')['dev_year']
                     .max()
                     .to_dict())


def event_probability_vector(dev):
    if dev not in event_probs_dict:
        dev = max(event_probs_dict.keys())
    row = event_probs_dict[dev]
    p = np.array([row[1], row[2], row[3], row[4]], dtype=float)
    p /= p.sum()
    return p


def severity_for(dev):
    if dev in severity_params:
        return severity_params[dev]['shape'], severity_params[dev]['scale']
    return severity_params_pooled['shape'], severity_params_pooled['scale']


open_records = open_claims.to_dict(orient='records')


def simulate_one_path():
    total = 0.0
    n_events = 0
    for claim in open_records:
        cid = claim['claim_id']
        start_dev = last_dev_observed.get(cid, 0) + 1
        for dev in range(start_dev, MAX_DEV + 1):
            p = event_probability_vector(dev)
            event = RNG.choice(EVENT_TYPES_ARR, p=p)
            n_events += 1
            if event in (2, 3):
                shape, scale = severity_for(dev)
                total += RNG.gamma(shape=shape, scale=scale)
            if event in (1, 2):
                break
    return total, n_events


sim_totals = np.empty(N_SIMULATIONS)
sim_event_counts = np.empty(N_SIMULATIONS, dtype=int)
for s in range(N_SIMULATIONS):
    sim_totals[s], sim_event_counts[s] = simulate_one_path()

mean = sim_totals.mean()
std = sim_totals.std(ddof=1)
percentiles = np.percentile(sim_totals, [50, 75, 95, 99, 99.5])

print(f"\n--- Section 5: ICR Stochastic Reserving — {N_SIMULATIONS} simulations ---")
print(f"Open claims projected: {len(open_records):,}")
print(f"Reserve distribution (EUR):")
print(f"  mean       : {mean:>15,.0f}")
print(f"  std        : {std:>15,.0f}")
print(f"  CV         : {std / mean:>15.3%}")
print(f"  median     : {percentiles[0]:>15,.0f}")
print(f"  75th pct   : {percentiles[1]:>15,.0f}")
print(f"  95th pct   : {percentiles[2]:>15,.0f}")
print(f"  99th pct   : {percentiles[3]:>15,.0f}")
print(f"  99.5% VaR  : {percentiles[4]:>15,.0f}")


--- Section 5: ICR Stochastic Reserving — 500 simulations ---
Open claims projected: 2,085
Reserve distribution (EUR):
  mean       :       4,198,906
  std        :         106,989
  CV         :          2.548%
  median     :       4,190,668
  75th pct   :       4,273,434
  95th pct   :       4,371,860
  99th pct   :       4,447,767
  99.5% VaR  :       4,469,641
